# Defining Churn

The dataset has no churn column, so we build one:

- A household has churned if it made **no purchase in the 28 days following a cutoff point T**.

### The trap

The approach of having label churn over the last 28 days (684-711) and
build features from the full history (1-711) silently breaks.
Recency measured at day 711 gives `recency > 28 -> churn = 1` with no exceptions.
The feature *is* the label. The model scores near-perfect AUC and has
learned nothing. This is **data leakage**. Spend and purchase counts are
contaminated the same way, more subtly.

### The fix: two non-overlapping windows

    days 1 ................ 683 | 684 ......... 711
    +-- observation window -----+ +-- outcome window --+
            FEATURES only              LABEL only
                              T = 683

Recency is now measured at T, and the identity breaks:

| household | last purchase <= 683 | recency_at_T | bought in 684-711? | churn |
|-----------|----------------------|--------------|--------------------|-------|
| 1042      | day 660              | 23           | no                 | 1     |
| 2088      | day 660              | 23           | yes (day 690)      | 0     |

Same feature, different labels. Recency stays a strong signal, but it is
now a clue rather than the answer.

**Working rule:** every feature aggregation carries `DAY <= 683`. Any
aggregation without it is assumed to leak until proven otherwise.

**Eligible population:** only households with at least one purchase in
the 90 days before T (594-683). Anyone quieter than that left long ago
and isn't churning now.

**Note:** T splits *time* to build a clean label. The train/test split
comes later and splits *households*. Different cuts, different purposes.

# Data prep

Initial exploration of the **8 raw datasets used in this project**.
The goal here is to load each table, inspect its structure (shape, keys, nulls, data types), apply light cleaning and make them ready for exploration

In [1]:
# Checking import happened correctly
import glob
hits = glob.glob("/kaggle/input/**/*.csv", recursive=True)
print(f"{len(hits)} CSV files found:")
for h in hits:
    print(" ", h)

8 CSV files found:
  /kaggle/input/datasets/frtgnn/dunnhumby-the-complete-journey/campaign_table.csv
  /kaggle/input/datasets/frtgnn/dunnhumby-the-complete-journey/causal_data.csv
  /kaggle/input/datasets/frtgnn/dunnhumby-the-complete-journey/coupon.csv
  /kaggle/input/datasets/frtgnn/dunnhumby-the-complete-journey/campaign_desc.csv
  /kaggle/input/datasets/frtgnn/dunnhumby-the-complete-journey/product.csv
  /kaggle/input/datasets/frtgnn/dunnhumby-the-complete-journey/transaction_data.csv
  /kaggle/input/datasets/frtgnn/dunnhumby-the-complete-journey/hh_demographic.csv
  /kaggle/input/datasets/frtgnn/dunnhumby-the-complete-journey/coupon_redempt.csv


In [2]:
# Define the path to access the databases
from pathlib import Path
RAW_PATH = Path("/kaggle/input/datasets/frtgnn/dunnhumby-the-complete-journey")

## Transactions data

**Data quality checks**

Before transforming the table, confirm that the assumed composite key (`household_key`, `BASKET_ID`, `PRODUCT_ID`) is actually unique, and check for missing values that would need to be handled.

In [3]:
# Load the transactions table and check its shape and first rows
import pandas as pd

transactions = pd.read_csv(RAW_PATH / "transaction_data.csv")
print(transactions.shape)
transactions.head()

(2595732, 12)


,household_key,BASKET_ID,DAY,PRODUCT_ID,QUANTITY,SALES_VALUE,STORE_ID,RETAIL_DISC,TRANS_TIME,WEEK_NO,COUPON_DISC,COUPON_MATCH_DISC
0,2375,26984851472,1,1004906,1,1.39,364,-0.60,1631,1,0.0,0.0
1,2375,26984851472,1,1033142,1,0.82,364,0.00,1631,1,0.0,0.0
2,2375,26984851472,1,1036325,1,0.99,364,-0.30,1631,1,0.0,0.0
3,2375,26984851472,1,1082185,1,1.21,364,0.00,1631,1,0.0,0.0
4,2375,26984851472,1,8160430,1,1.50,364,-0.39,1631,1,0.0,0.0


In [4]:
# Map the interval of days
print(transactions["DAY"].min(), transactions["DAY"].max())

# Map the Number of HH.
print(transactions["household_key"].nunique())

1 711
2500


In [5]:
# Verify that household_key + BASKET_ID + PRODUCT_ID together form a unique key (no duplicate rows for the same product in the same basket)
key_cols = ["household_key", "BASKET_ID", "PRODUCT_ID"]
print(transactions.shape[0])
print(transactions.duplicated(subset=key_cols).sum())

2595732
0


In [6]:
# Check for missing values in every column
transactions.isnull().sum()

household_key        0
BASKET_ID            0
DAY                  0
PRODUCT_ID           0
QUANTITY             0
SALES_VALUE          0
STORE_ID             0
RETAIL_DISC          0
TRANS_TIME           0
WEEK_NO              0
COUPON_DISC          0
COUPON_MATCH_DISC    0
dtype: int64

## Product Table
Product catalog with department, brand and commodity/sub-commodity descriptions, keyed by PRODUCT_ID.

In [7]:
# Load the product table and confirm PRODUCT_ID is a unique key
product = pd.read_csv(RAW_PATH / "product.csv")
print(product.shape)
# Confirming Key
print(product["PRODUCT_ID"].duplicated().sum())
product.head()

(92353, 7)
0


,PRODUCT_ID,MANUFACTURER,DEPARTMENT,BRAND,COMMODITY_DESC,SUB_COMMODITY_DESC,CURR_SIZE_OF_PRODUCT
0,25671,2,GROCERY,National,FRZN ICE,ICE - CRUSHED/CUBED,22 LB
1,26081,2,MISC. TRANS.,National,NO COMMODITY DESCRIPTION,NO SUBCOMMODITY DESCRIPTION,
2,26093,69,PASTRY,Private,BREAD,BREAD:ITALIAN/FRENCH,
3,26190,69,GROCERY,Private,FRUIT - SHELF STABLE,APPLE SAUCE,50 OZ
4,26355,69,GROCERY,Private,COOKIES/CONES,SPECIALTY COOKIES,14 OZ


## Household demographics
Household-level demographic attributes (age, income, marital status, household composition, etc.), keyed by household_key.

In [8]:
# Load the household demographic table and confirm there are no fully duplicated rows
hh_demographic = pd.read_csv(RAW_PATH / "hh_demographic.csv")
print(hh_demographic.shape)
print(hh_demographic["household_key"].duplicated().sum())
hh_demographic.head()

(801, 8)
0


,AGE_DESC,MARITAL_STATUS_CODE,INCOME_DESC,HOMEOWNER_DESC,HH_COMP_DESC,HOUSEHOLD_SIZE_DESC,KID_CATEGORY_DESC,household_key
0,65+,A,35-49K,Homeowner,2 Adults No Kids,2,None/Unknown,1
1,45-54,A,50-74K,Homeowner,2 Adults No Kids,2,None/Unknown,7
2,25-34,U,25-34K,Unknown,2 Adults Kids,3,1,8
3,25-34,U,75-99K,Homeowner,2 Adults Kids,4,2,13
4,45-54,B,50-74K,Homeowner,Single Female,1,None/Unknown,16


In [9]:
# Check for missing values in every column
hh_demographic.isnull().sum()

AGE_DESC               0
MARITAL_STATUS_CODE    0
INCOME_DESC            0
HOMEOWNER_DESC         0
HH_COMP_DESC           0
HOUSEHOLD_SIZE_DESC    0
KID_CATEGORY_DESC      0
household_key          0
dtype: int64

In [10]:
for c in hh_demographic.select_dtypes("object"):
    print(c, hh_demographic[c].value_counts().to_dict())

AGE_DESC {'45-54': 288, '35-44': 194, '25-34': 142, '65+': 72, '55-64': 59, '19-24': 46}
MARITAL_STATUS_CODE {'U': 344, 'A': 340, 'B': 117}
INCOME_DESC {'50-74K': 192, '35-49K': 172, '75-99K': 96, '25-34K': 77, '15-24K': 74, 'Under 15K': 61, '125-149K': 38, '100-124K': 34, '150-174K': 30, '250K+': 11, '175-199K': 11, '200-249K': 5}
HOMEOWNER_DESC {'Homeowner': 504, 'Unknown': 233, 'Renter': 42, 'Probable Renter': 11, 'Probable Owner': 11}
HH_COMP_DESC {'2 Adults No Kids': 255, '2 Adults Kids': 187, 'Single Female': 144, 'Single Male': 95, 'Unknown': 73, '1 Adult Kids': 47}
HOUSEHOLD_SIZE_DESC {'2': 318, '1': 255, '3': 109, '5+': 66, '4': 53}
KID_CATEGORY_DESC {'None/Unknown': 558, '1': 114, '3+': 69, '2': 60}


**Inferring missing kid category**

`KID_CATEGORY_DESC` is `'None/Unknown'` for a large share of households, which conflates two very different cases: households that genuinely have no kids, and households where the number of kids simply wasn't captured. Household composition (`HH_COMP_DESC`) lets us recover some of that signal — categories like `'2 Adults No Kids'`, `'Single Female'` and `'Single Male'` imply no kids even when `KID_CATEGORY_DESC` doesn't say so explicitly. Use this to build a cleaner `KID_CATEGORY_INFERRED` column.

In [11]:
pd.crosstab(hh_demographic["HH_COMP_DESC"], hh_demographic["KID_CATEGORY_DESC"])

KID_CATEGORY_DESC,1,2,3+,None/Unknown
HH_COMP_DESC,,,,
1 Adult Kids,20,14,13,0
2 Adults Kids,88,46,53,0
2 Adults No Kids,0,0,0,255
Single Female,0,0,0,144
Single Male,0,0,0,95
Unknown,6,0,3,64


In [12]:
# KID_CATEGORY_DESC is 'None/Unknown' for many households where we can still infer there are no kids
# from HH_COMP_DESC (e.g. '2 Adults No Kids', 'Single Female', 'Single Male').
# Build a cleaner KID_CATEGORY_INFERRED column: keep the original value when it is known,
# set it to 'None' when the household composition implies no kids, otherwise leave it as 'Unknown'.
import numpy as np

no_kids_categories = ["2 Adults No Kids", "Single Female", "Single Male"]

hh_demographic["KID_CATEGORY_INFERRED"] = np.select(
    [
        hh_demographic["KID_CATEGORY_DESC"] != "None/Unknown",
        hh_demographic["HH_COMP_DESC"].isin(no_kids_categories),
    ],
    [
        hh_demographic["KID_CATEGORY_DESC"],
        "None",
    ],
    default="Unknown",
)

In [13]:
pd.crosstab(hh_demographic["HH_COMP_DESC"], hh_demographic["KID_CATEGORY_INFERRED"])

KID_CATEGORY_INFERRED,1,2,3+,None,Unknown
HH_COMP_DESC,,,,,
1 Adult Kids,20,14,13,0,0
2 Adults Kids,88,46,53,0,0
2 Adults No Kids,0,0,0,255,0
Single Female,0,0,0,144,0
Single Male,0,0,0,95,0
Unknown,6,0,3,0,64


In [14]:
# Check the resulting distribution of the inferred kid category
hh_demographic["KID_CATEGORY_INFERRED"].value_counts()

KID_CATEGORY_INFERRED
None       494
1          114
3+          69
Unknown     64
2           60
Name: count, dtype: int64

### Testing for correlatione between datasets

In [15]:
# transactions -> product
products_not_on_master = set(transactions["PRODUCT_ID"].unique()) - set(product["PRODUCT_ID"].unique())

# transactions -> hh_demographic
homes_not_on_master = set(transactions["household_key"].unique()) - set(hh_demographic["household_key"].unique())

print(f"Products from Transaction not on Product Table: {len(products_not_on_master)}")
print(f"Houses from Transaction not on Houses Table: {len(homes_not_on_master)}")

Products from Transaction not on Product Table: 0
Houses from Transaction not on Houses Table: 1699


## 